In [0]:
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from databricks.sdk import WorkspaceClient

APP_NAME   = "region-signal"
MAILER_URL = "https://script.google.com/macros/s/AKfycbydL3ajCHec_0suJmTkweJfrVr_w0YCTmIHXC7adae21gPb3DBRiizvlbPdcY8yBglC/exec"
SECRET     = "change-this-to-a-long-random-text"
RECIPIENTS = "vinoth.shankar@tigeranalytics.com"      # comma-separated for more users

w = WorkspaceClient()

def compute_state():
    s = w.apps.get(APP_NAME).compute_status.state
    return getattr(s, "value", str(s))

# If it is already running, stop it first so the 3-hour window restarts now
if compute_state() == "ACTIVE":
    print("App already running - restarting to reset the window")
    w.apps.stop(APP_NAME).result()
w.apps.start(APP_NAME).result()
app_url = w.apps.get(APP_NAME).url

now = datetime.now(ZoneInfo("America/Chicago"))
fmt = lambda d: d.strftime("%a %b %d, %I:%M %p CT")
html = f"""
<p>Hello,</p>
<p>The <b>Pepsico Forecast Engine - Market Signal</b> app is now up and running.</p>
<p><a href="{app_url}" style="background:#1b6ef3;color:#fff;padding:10px 18px;border-radius:6px;text-decoration:none;font-weight:bold">Open the app</a></p>
<ul>
 <li>Sign in with your user ID and password.</li>
 <li>All markets are checked by default (use this week's forecast).</li>
 <li>Uncheck any market that should fall back to the previous valid forecast.</li>
 <li>Press <b>Submit</b>. Nothing is saved until you do.</li>
</ul>
<p>The app is open for <b>3 hours</b>, from {fmt(now)} to {fmt(now + timedelta(hours=3))}. After that, updates are blocked and the last submitted selections are used for the O9 load.</p>
<p>Need help? Contact forecast_region_admin@pepsico.com</p>
<p>Thank you,<br>Forecast Engine - Market Signal</p>"""

r = requests.post(MAILER_URL, json={"secret": SECRET, "to": RECIPIENTS,
    "subject": "ACTION REQUIRED: Pepsico Forecast Engine - Market Signal is now open (3 hours)",
    "html": html}, timeout=30)
print(r.status_code, r.text[:200])
assert r.text.strip() == "sent", "Email was not sent: " + r.text[:200]